In [3]:
from operator import add
from typing import TypedDict, Annotated
from rich import print as rprint
from langgraph.graph import StateGraph, START, END


class OverALlState(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverALlState) -> OverALlState:
    for k,v in state.items():
        print(f"k:{k} v:{v}")

    return state

builder = StateGraph(state_schema=OverALlState)
builder.add_node("node_1", node_1)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

graph = builder.compile()
result = graph.invoke({"cur_id": "start"})

rprint(result)


k:logs v:[]
k:cur_id v:start


{'logs': [], 'cur_id': 'start'}

In [4]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Overwrite
from typing import TypedDict, Annotated
from operator import add

class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    id: str

def node_a(state: OverAllState):
    return {
        "logs": ["node_a"],
        "id": "node_a"
    }

def node_b(state: OverAllState):
    return {
        "logs": Overwrite(["node_b"]),
        "id": "node_b"
    }

def node_c(state: OverAllState):
    return {
        "logs": ["node_c"],
        "id": "node_c"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_node("node_c", node_c)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", "node_c")
builder.add_edge("node_c", END)

graph = builder.compile()
result = graph.invoke({"logs": ["START"], "id": "start"})
print('=' * 30, '-> result <-', '=' * 30)
print(result)

============================== -> result <- ==============================
{'logs': ['node_b', 'node_c'], 'id': 'node_c'}
